In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime

CATALOG     = "olist_ecommerce"
SILVER      = f"{CATALOG}.silver"
GOLD        = f"{CATALOG}.gold"
RUN_ID      = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Gold layer started — Run ID: {RUN_ID}")
print(f"Silver: {SILVER}")
print(f"Gold:   {GOLD}")

In [0]:
def apply_scd2(source_df, target_table, natural_key,
               tracked_cols, surrogate_col):
    """
    SCD Type 2 implementation using Delta MERGE INTO.

    Logic:
    - Natural key không tồn tại → INSERT mới (is_current=True)
    - Natural key tồn tại + tracked cols thay đổi →
        UPDATE cũ (is_current=False, valid_to=now)
        INSERT mới (is_current=True, valid_from=now)
    - Natural key tồn tại + không thay đổi → NOTHING
    """

    # Thêm SCD2 metadata columns
    source_with_meta = (source_df
        .withColumn(surrogate_col,   expr("uuid()"))
        .withColumn("valid_from",    current_timestamp())
        .withColumn("valid_to",
            lit("9999-12-31 00:00:00").cast("timestamp"))
        .withColumn("is_current",    lit(True))
        .withColumn("_updated_at",   current_timestamp()))

   
    # Lần đầu tiên — tạo table mới
    if not spark.catalog.tableExists(target_table):
        (source_with_meta.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table))
        count = spark.read.table(target_table).count()
        print(f"[SCD2] Initial load {target_table}: {count:,} records")
        return

    # Build change detection condition
    change_cond = " OR ".join([
        f"target.{c} != source.{c} OR "
        f"(target.{c} IS NULL AND source.{c} IS NOT NULL)"
        for c in tracked_cols
    ])

    delta_tbl = DeltaTable.forName(spark, target_table)

    # Step 1: Expire bản ghi cũ khi có thay đổi
    (delta_tbl.alias("target")
        .merge(
            source_with_meta.alias("source"),
            f"target.{natural_key} = source.{natural_key} "
            f"AND target.is_current = true")
        .whenMatchedUpdate(
            condition=change_cond,
            set={
                "valid_to":   "current_timestamp()",
                "is_current": "false"
            })
        .execute())

    # Step 2: Insert version mới cho những key chưa có
    # hoặc vừa bị expire ở Step 1
    existing_current = (spark.read.table(target_table)
        .filter("is_current = true")
        .select(natural_key))

    new_versions = source_with_meta.join(
        existing_current, natural_key, "left_anti")

    if new_versions.count() > 0:
        (new_versions.write
            .format("delta")
            .mode("append")
            .saveAsTable(target_table))
        print(f"[SCD2] Inserted {new_versions.count():,} "
              f"new versions into {target_table}")
    else:
        print(f"[SCD2] No changes detected in {target_table}")

In [0]:
print("Building dim_customer...")

silver_customers = spark.read.table(f"{SILVER}.customers")

# Select columns cần thiết cho dim
dim_customer_src = silver_customers.select(
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "brazil_region",
    "geo_lat",
    "geo_lng"
).dropDuplicates(["customer_unique_id"])

print(f"Source rows: {dim_customer_src.count():,}")

# Apply SCD2
apply_scd2(
    source_df     = dim_customer_src,
    target_table  = f"{GOLD}.dim_customer",
    natural_key   = "customer_unique_id",
    tracked_cols  = ["customer_city", "customer_state"],
    surrogate_col = "customer_key"
)

# Verify
spark.sql(f"""
    SELECT is_current,
           COUNT(*) AS count
    FROM {GOLD}.dim_customer
    GROUP BY is_current
""").show()

spark.sql(f"""
    SELECT customer_key,
           customer_unique_id,
           customer_city,
           customer_state,
           valid_from,
           valid_to,
           is_current
    FROM {GOLD}.dim_customer
    LIMIT 5
""").show(truncate=False)

In [0]:
print("Building dim_product...")

silver_products = spark.read.table(f"{SILVER}.products")

dim_product_src = silver_products.select(
    "product_id",
    "product_category_name_english",
    "weight_kg",
    "volume_cm3",
    "size_tier",
    "product_photos_qty",
    "listing_quality_score"
).dropDuplicates(["product_id"])

print(f"Source rows: {dim_product_src.count():,}")

apply_scd2(
    source_df     = dim_product_src,
    target_table  = f"{GOLD}.dim_product",
    natural_key   = "product_id",
    tracked_cols  = ["product_category_name_english",
                     "weight_kg", "size_tier"],
    surrogate_col = "product_key"
)

# Verify
spark.sql(f"""
    SELECT is_current, COUNT(*) AS count
    FROM {GOLD}.dim_product
    GROUP BY is_current
""").show()

print("dim_product done!")

In [0]:
print("Building dim_seller...")

silver_sellers = spark.read.table(f"{SILVER}.sellers")

dim_seller_src = silver_sellers.select(
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
    "brazil_region",
    "geo_lat",
    "geo_lng"
).dropDuplicates(["seller_id"])

print(f"Source rows: {dim_seller_src.count():,}")

apply_scd2(
    source_df     = dim_seller_src,
    target_table  = f"{GOLD}.dim_seller",
    natural_key   = "seller_id",
    tracked_cols  = ["seller_city", "seller_state"],
    surrogate_col = "seller_key"
)

# Verify
spark.sql(f"""
    SELECT is_current, COUNT(*) AS count
    FROM {GOLD}.dim_seller
    GROUP BY is_current
""").show()

print("dim_seller done!")

In [0]:
print("Building dim_date...")

# Generate date range 2016 → 2026
dim_date = (spark.sql("""
    SELECT explode(
        sequence(
            to_date('2016-01-01'),
            to_date('2026-12-31'),
            interval 1 day
        )
    ) AS calendar_date
""")
    .withColumn("date_key",
        date_format("calendar_date", "yyyyMMdd").cast("int"))
    .withColumn("year",
        year("calendar_date"))
    .withColumn("quarter",
        quarter("calendar_date"))
    .withColumn("month",
        month("calendar_date"))
    .withColumn("month_name",
        date_format("calendar_date", "MMMM"))
    .withColumn("week_of_year",
        weekofyear("calendar_date"))
    .withColumn("day_of_week",
        dayofweek("calendar_date"))
    .withColumn("day_name",
        date_format("calendar_date", "EEEE"))
    .withColumn("is_weekend",
        dayofweek("calendar_date").isin([1, 7]))
    .withColumn("is_weekday",
        ~dayofweek("calendar_date").isin([1, 7]))
)

(dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.dim_date"))

count = spark.read.table(f"{GOLD}.dim_date").count()
print(f"dim_date: {count:,} rows (2016-2026)")

spark.sql(f"""
    SELECT date_key, calendar_date, year,
           month_name, day_name, is_weekend
    FROM {GOLD}.dim_date
    WHERE calendar_date = '2018-11-23'
""").show()

In [0]:
"""
Test SCD2 bằng cách simulate customer đổi city.
Đây là test QUAN TRỌNG nhất — interviewer sẽ hỏi cái này.
"""

print("=" * 55)
print("SCD2 VERIFICATION TEST")
print("=" * 55)

# Lấy 1 customer để test
test_customer = (spark.read.table(f"{GOLD}.dim_customer")
    .filter("is_current = true")
    .limit(1)
    .select("customer_unique_id", "customer_city",
            "customer_state")
    .collect()[0])

cust_id   = test_customer["customer_unique_id"]
old_city  = test_customer["customer_city"]
old_state = test_customer["customer_state"]

print(f"\nTest customer: {cust_id}")
print(f"Current city: {old_city}, state: {old_state}")

# Tạo "update" — simulate customer đổi city
updated_customer = spark.createDataFrame([{
    "customer_unique_id":      cust_id,
    "customer_zip_code_prefix": "99999",
    "customer_city":            "Test City Updated",
    "customer_state":           old_state,
    "brazil_region":            "Sudeste",
    "geo_lat":                  -23.5,
    "geo_lng":                  -46.6
}])

print("\nApplying SCD2 update...")

# Apply SCD2 với data mới
apply_scd2(
    source_df     = updated_customer,
    target_table  = f"{GOLD}.dim_customer",
    natural_key   = "customer_unique_id",
    tracked_cols  = ["customer_city", "customer_state"],
    surrogate_col = "customer_key"
)

# Verify: customer này phải có 2 versions
print(f"\nHistory for customer {cust_id}:")
spark.sql(f"""
    SELECT customer_key,
           customer_city,
           customer_state,
           valid_from,
           valid_to,
           is_current
    FROM {GOLD}.dim_customer
    WHERE customer_unique_id = '{cust_id}'
    ORDER BY valid_from
""").show(truncate=False)

# Phải thấy:
# Row 1: old_city, is_current=FALSE, valid_to=now
# Row 2: "Test City Updated", is_current=TRUE, valid_to=9999
print("\n✅ SCD2 working correctly if you see 2 rows above!")
print("   Row 1: old version (is_current=FALSE)")
print("   Row 2: new version (is_current=TRUE)")

In [0]:
print("=" * 55)
print("GOLD LAYER — DIM TABLES SUMMARY")
print("=" * 55)

spark.sql(f"""
    SELECT * FROM (
        SELECT 'dim_customer' AS table_name,
               COUNT(*)       AS total_rows,
               SUM(CASE WHEN is_current THEN 1 ELSE 0 END)
                              AS current_rows
        FROM {GOLD}.dim_customer UNION ALL

        SELECT 'dim_product',
               COUNT(*),
               SUM(CASE WHEN is_current THEN 1 ELSE 0 END)
        FROM {GOLD}.dim_product UNION ALL

        SELECT 'dim_seller',
               COUNT(*),
               SUM(CASE WHEN is_current THEN 1 ELSE 0 END)
        FROM {GOLD}.dim_seller UNION ALL

        SELECT 'dim_date',
               COUNT(*),
               COUNT(*)
        FROM {GOLD}.dim_date
    )
    ORDER BY total_rows DESC
""").show()

print("\n✅ All dims ready! Next: Notebook 04 — fact_orders streaming")